In [ ]:
import requests
import json

url = "https://opensearch.pads.fim.uni-passau.de/msmarco-v2.1-segmented/_search"
auth = ("hana", "4DBpreroRMc!fPPczxaJ")
query = "Restaurants in Passau"

params = {
    "query": {"match": {"segment": query}},
    "size": 50,  # top-50
    "_source": ["docid", "segment"]  # Extract docid + text
}

resp = requests.get(url, auth=auth, json=params)
hits = resp.json()["hits"]["hits"]
scores = [h["_score"] for h in hits]
docids = [h["_source"]["docid"] for h in hits]

In [ ]:
hits

In [9]:
import ir_datasets
dataset = ir_datasets.load("msmarco-passage/train/judged")

In [3]:
dataset.queries_iter()

<generator object FilteredQueries.queries_iter at 0x000001F3234E52A0>

In [ ]:
for query in dataset.queries_iter():
    print(query) # namedtuple<query_id, text>

In [13]:

for qrel in dataset.qrels_iter():
    if qrel.relevance == 0:
        print(qrel.relevance)

In [ ]:
import pandas as pd
from collections import defaultdict

# Group qrels by query_id
qrels_by_query = defaultdict(list)
for qrel in dataset.qrels_iter():
    qrels_by_query[qrel.query_id].append(qrel)

# Filter queries with >5 qrels
queries_with_5plus = {qid: qrels for qid, qrels in qrels_by_query.items() 
                      if len(qrels) > 5}

# Create CSV data
csv_data = []
for query_id, qrels in queries_with_5plus.items():
    for qrel in qrels:
        csv_data.append({
            'query_id': str(qrel.query_id),
            'doc_id': str(qrel.doc_id),
            'relevance': int(qrel.relevance),
            'iteration': str(getattr(qrel, 'iteration', '0'))
        })

# Save to CSV
df = pd.DataFrame(csv_data)
df.to_csv('msmarco_queries_5plus_qrels.csv', index=False)

print(f"Saved {len(csv_data)} qrels from {len(queries_with_5plus)} queries")
print(f"\nQrels per query stats:")
print(df.groupby('query_id').size().describe())